# 03 · Preprocesamiento — PySpark

**Autores:** Santiago Hurtado, Juan Marín, Andrés Parejo

Pipeline de PySpark **independiente** del de scikit-learn: vuelve a leer el
CSV crudo (no el parquet ya limpio de pandas), con su propio
`StringIndexer`/`OneHotEncoder`/`VectorAssembler`. En ningún momento se usa
`.toPandas()` ni `.collect()` sobre el DataFrame completo — solo sobre
agregados pequeños (conteos por categoría para detectar categorías raras).

Ejecutado **localmente** (PC propio: Ryzen 5 5600G / 64GB RAM / Java 17),
en vez de Google Colab.

## 1. Java + utils.py local

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import utils

java_home = utils.setup_java_home()
hadoop_home = utils.setup_hadoop_home()
print("JAVA_HOME:", java_home)
print("HADOOP_HOME:", hadoop_home)

JAVA_HOME: C:\Users\boolean\Downloads\requirements fin\Copy of requirements/tools\jdk-17.0.20.1+1
HADOOP_HOME: C:\Users\boolean\Downloads\requirements fin\Copy of requirements/tools/hadoop


In [2]:
import shutil
import gzip
import time

assert os.path.exists(utils.RAW_CSV_LOCAL), f"No se encontró el CSV en {utils.RAW_CSV_LOCAL}"
print(f"{os.path.getsize(utils.RAW_CSV_LOCAL) / 1e9:.2f} GB listo en {utils.RAW_CSV_LOCAL}")

1.68 GB listo en C:\Users\boolean\Downloads\archive\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv


## 2. Configuración de la sesión de Spark

El enunciado exige literalmente este bloque:

```python
spark = SparkSession.builder \
    .appName("LendingClub_Optimized") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.memory.fraction", 0.8) \
    .config("spark.memory.storageFraction", 0.3) \
    .getOrCreate()
```

En **modo local** (`local[*]`, sin cluster real) `spark.executor.memory` no
tiene efecto: driver y "executor" comparten la misma JVM, y lo único que
realmente limita la memoria disponible es `spark.driver.memory`. Además,
`400` particiones de shuffle tiene sentido para un cluster con decenas de
cores — en esta máquina (unos pocos/varios vCPUs) 400 particiones para
~2.26M filas generarían particiones diminutas y un overhead de
planificación de tareas desproporcionado. Por eso se usa la config de abajo,
calculada dinámicamente a partir de los recursos reales de esta sesión
(`spark.driver.memory`/`shuffle.partitions`/`parallelism` "equivalentes
debidamente justificados", tal como el enunciado permite explícitamente).
El resto de parámetros (`memory.fraction`, `memory.storageFraction`) sí se
mantienen literales, porque son fracciones/proporciones, no tamaños
absolutos — no dependen del hardware.

In [3]:
driver_mem_gb, shuffle_partitions, res = utils.get_spark_driver_config()
print(f"Recursos detectados: {res}")
print(f"-> spark.driver.memory = {driver_mem_gb}g | shuffle.partitions/parallelism = {shuffle_partitions}")

Recursos detectados: {'cpu_count': 12, 'total_ram_gb': 63.3, 'free_ram_gb': 52.49}
-> spark.driver.memory = 32g | shuffle.partitions/parallelism = 36


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LendingClub_Optimized")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
    .config("spark.default.parallelism", str(shuffle_partitions))
    .config("spark.driver.memory", f"{driver_mem_gb}g")
    .config("spark.executor.memory", f"{driver_mem_gb}g")  # no-op en local[*], se deja por completitud
    .config("spark.memory.fraction", 0.8)
    .config("spark.memory.storageFraction", 0.3)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 3. Leer el CSV completo con Spark

Se lee primero con todas las columnas como `string` (sin `inferSchema`, que
forzaría un pase completo extra solo para adivinar tipos) y se
selecciona/castea explícitamente el mismo subconjunto de 24 columnas +
`loan_status` usado del lado de scikit-learn — así ambos frameworks
entrenan sobre exactamente las mismas variables, y la comparación de
desempeño mide el framework, no diferencias de features.

In [5]:
t0 = time.time()
raw = spark.read.csv(f"file:///{utils.RAW_CSV_LOCAL.replace(chr(92), '/')}", header=True, inferSchema=False)
t_read = time.time() - t0
print(f"Lectura (lazy) declarada en {t_read:.2f}s | columnas originales: {len(raw.columns)}")

Lectura (lazy) declarada en 2.77s | columnas originales: 151


In [6]:
sdf = raw.select(*utils.RAW_USECOLS)

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

numeric_raw_cols = ["dti", "revol_util", "mort_acc", "annual_inc", "pub_rec",
                     "delinq_2yrs", "fico_range_high", "fico_range_low", "int_rate",
                     "installment", "open_acc", "total_acc", "revol_bal", "loan_amnt"]
for c in numeric_raw_cols:
    sdf = sdf.withColumn(c, F.col(c).cast(DoubleType()))

sdf = sdf.withColumn("term", F.trim(F.col("term")))

t0 = time.time()
n_rows_raw = sdf.count()  # accion pequena (un entero), no una recoleccion del dataset
t_count = time.time() - t0
print(f"Filas leidas: {n_rows_raw:,} en {t_count:.2f}s (accion real que dispara la lectura completa)")
assert n_rows_raw > 1_300_000, "Menos de 1.3M filas -- viola la condicion obligatoria del dataset completo"

Filas leidas: 2,260,701 en 1.40s (accion real que dispara la lectura completa)


## 4. Limpieza: filas vacías, target, `emp_length`, nulos, categorías raras

Réplica en Spark de las mismas decisiones tomadas en pandas
(`00_setup_datos.ipynb`), para que ambos frameworks entrenen sobre datos
equivalentes.

In [7]:
sdf = sdf.filter(F.col("loan_status").isNotNull())

sdf = sdf.withColumn("default", F.when(F.col("loan_status") == "Charged Off", 1).otherwise(0).cast("double"))

emp_length_map = utils.EMP_LENGTH_MAP
emp_expr = F.when(F.col("emp_length").isNull(), None)
for k, v in emp_length_map.items():
    emp_expr = emp_expr.when(F.col("emp_length") == k, float(v))
sdf = sdf.withColumn("emp_length_num", emp_expr.otherwise(None).cast(DoubleType()))

print("Filas tras quitar vacías:", sdf.count())

Filas tras quitar vacías: 2260668


Imputación de nulos numéricos con la mediana vía `pyspark.ml.feature.Imputer`
(equivalente a `fillna(median())` en pandas, pero calculado de forma
distribuida sin traer la columna al driver).

In [8]:
from pyspark.ml.feature import Imputer

impute_cols = numeric_raw_cols + ["emp_length_num"]
imputer = Imputer(strategy="median", inputCols=impute_cols, outputCols=impute_cols)
sdf = imputer.fit(sdf).transform(sdf)

Agrupar categorías raras (<1%) en `purpose`, `home_ownership`, `addr_state`
— mismo criterio que en pandas. `groupBy().count()` es una agregación
pequeña (a lo sumo ~50 filas de salida), no una recolección del dataset
completo, así que el `.collect()` sobre *eso* sí está permitido.

In [9]:
n_total = sdf.count()
rare_label = {"purpose": "other", "home_ownership": "OTHER", "addr_state": "Other"}

for col in ["purpose", "home_ownership", "addr_state"]:
    counts = sdf.groupBy(col).count().collect()  # agregado pequeño, no el dataset completo
    rare_values = [row[col] for row in counts if row["count"] / n_total < utils.RARE_CATEGORY_THRESHOLD]
    sdf = sdf.withColumn(
        col, F.when(F.col(col).isin(rare_values), rare_label[col]).otherwise(F.col(col))
    )
    print(f"{col}: {len(rare_values)} categorías agrupadas en '{rare_label[col]}'")

purpose: 261 categorías agrupadas en 'other'


home_ownership: 4 categorías agrupadas en 'OTHER'


addr_state: 243 categorías agrupadas en 'Other'


## 5. StringIndexer + OneHotEncoder

In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder as SparkOHE, VectorAssembler, StandardScaler as SparkScaler

categorical_cols = utils.CATEGORICAL_COLS
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [SparkOHE(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

## 6. VectorAssembler + StandardScaler

In [11]:
numeric_cols_spark = utils.NUMERIC_COLS
assembler_inputs = numeric_cols_spark + [f"{c}_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw")
scaler = SparkScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

## 7. Ejecutar el pipeline de transformación y cachear

`.persist(StorageLevel.MEMORY_AND_DISK)` es obligatorio por el enunciado.
Se usa `MEMORY_AND_DISK` (no solo `.cache()`, que es memoria pura) porque
aunque esta máquina tiene bastante RAM, es la misma política robusta usada
en Colab: si no cabe todo en memoria, Spark derrama a disco en vez de
fallar.

In [12]:
from pyspark.ml import Pipeline
from pyspark import StorageLevel

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

t0 = time.time()
prep_model = prep_pipeline.fit(sdf)
sdf_features = prep_model.transform(sdf).select("features", "default")
sdf_features = sdf_features.persist(StorageLevel.MEMORY_AND_DISK)
n_cached = sdf_features.count()  # fuerza la materialización del cache (accion pequena: un entero)
t_pipeline = time.time() - t0

print(f"Pipeline de preprocesamiento (fit+transform+persist) en {t_pipeline:.2f}s | filas cacheadas: {n_cached:,}")

Pipeline de preprocesamiento (fit+transform+persist) en 46.82s | filas cacheadas: 2,260,668


## 8. División train/test 80/20 estratificada

`randomSplit` no tiene un parámetro `stratify=` nativo como scikit-learn.
Para lograr el mismo efecto, se separa por clase, se aplica `randomSplit`
a cada una por separado y se unen — así la proporción de `default` queda
igual en train y test, igual que con `stratify=y` en pandas.

In [13]:
t0 = time.time()

class0 = sdf_features.filter(F.col("default") == 0.0)
class1 = sdf_features.filter(F.col("default") == 1.0)

train0, test0 = class0.randomSplit([0.8, 0.2], seed=42)
train1, test1 = class1.randomSplit([0.8, 0.2], seed=42)

train_df = train0.union(train1).persist(StorageLevel.MEMORY_AND_DISK)
test_df = test0.union(test1).persist(StorageLevel.MEMORY_AND_DISK)

n_train = train_df.count()
n_test = test_df.count()
t_split = time.time() - t0

print(f"Train: {n_train:,} | Test: {n_test:,} | split estratificado en {t_split:.2f}s")

Train: 1,807,851 | Test: 452,817 | split estratificado en 21.43s


## 9. Guardar splits y metadatos

Se guardan como Parquet (formato nativo de Spark, columnar) para que
`05_modeling_spark.ipynb` no tenga que rehacer todo este pipeline.

In [14]:
train_df.write.mode("overwrite").parquet(f"{utils.DATA_DIR}/spark_train.parquet")
test_df.write.mode("overwrite").parquet(f"{utils.DATA_DIR}/spark_test.parquet")

timing = {
    "read_declare_seconds": round(t_read, 3),
    "count_after_read_seconds": round(t_count, 3),
    "fit_transform_persist_seconds": round(t_pipeline, 3),
    "stratified_split_seconds": round(t_split, 3),
    "n_rows": int(n_rows_raw), "n_train": int(n_train), "n_test": int(n_test),
    "n_features_assembled": len(assembler_inputs),
    "spark_config": {
        "driver_memory_gb": driver_mem_gb, "shuffle_partitions": shuffle_partitions,
        "detected_resources": res,
    },
}
utils.save_json(timing, f"{utils.RESULTS_DIR}/03_spark_preprocessing_timing.json")
print(timing)

{'read_declare_seconds': 2.77, 'count_after_read_seconds': 1.405, 'fit_transform_persist_seconds': 46.822, 'stratified_split_seconds': 21.427, 'n_rows': 2260701, 'n_train': 1807851, 'n_test': 452817, 'n_features_assembled': 23, 'spark_config': {'driver_memory_gb': 32, 'shuffle_partitions': 36, 'detected_resources': {'cpu_count': 12, 'total_ram_gb': 63.3, 'free_ram_gb': 52.49}}}


## 10. Verificación

In [15]:
train_df.printSchema()
print("Proporción de default en train:", train_df.filter(F.col("default") == 1.0).count() / n_train)
print("Proporción de default en test:", test_df.filter(F.col("default") == 1.0).count() / n_test)

root
 |-- features: vector (nullable = true)
 |-- default: double (nullable = false)



Proporción de default en train: 0.11884939632746283


Proporción de default en test: 0.11858212037092247
